# Music Annotation Project V4: Incremental Dataset Builder and Version Publisher

**Runtime:** CPU.

This notebook preserves the V3.2 incremental page builder and adds immutable, content-addressed dataset publication. Changed pages produce new uncompressed TAR page objects; unchanged page objects are reused. A small version manifest records the exact page objects in each dataset release. No monolithic training ZIP is created.

The HITL split assignment file remains authoritative. Do not delete it casually, or historical HITL pages may receive different assignments.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

In [ ]:
import cv2, json, hashlib, re, shutil, tarfile
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

CONFIG_PATH = Path('/content/drive/MyDrive/MusicAnnotationProject/project_config.json')
if not CONFIG_PATH.exists():
    raise RuntimeError(f'Missing {CONFIG_PATH}')
CONFIG = json.loads(CONFIG_PATH.read_text())
ROOT = Path(CONFIG['project_root'])

ALIGNED_DIR = ROOT/'aligned'
CORRECTED_MASK_DIR = ROOT/'masks'/'corrected'
TRAINING_MASK_DIR = ROOT/'masks'/'training'
DATASET_DIR = ROOT/'dataset'
TRAIN_IMAGES = DATASET_DIR/'images'/'train'
VAL_IMAGES = DATASET_DIR/'images'/'val'
TRAIN_LABELS = DATASET_DIR/'labels'/'train'
VAL_LABELS = DATASET_DIR/'labels'/'val'
TRAIN_MASK_TILES = DATASET_DIR/'mask_tiles'/'train'
VAL_MASK_TILES = DATASET_DIR/'mask_tiles'/'val'
CORRECTED_REVIEW = ROOT/'review'/'corrected_masks'
TRAINING_REVIEW = ROOT/'review'/'training_masks'
TILE_REVIEW = ROOT/'review'/'tiles'
HITL_AUDIT_DIR = ROOT/'human_in_the_middle'/'audit'
STATE_DIR = ROOT/'pipeline_state'
STATE_PATH = STATE_DIR/'notebook1_dataset_state.json'
HITL_SPLIT_PATH = STATE_DIR/'hitl_split_assignments.json'
YAML = ROOT/'dataset_v3.yaml'
MANIFEST = ROOT/'dataset_manifest_v3.json'

# V4 immutable publication paths.
DATASET_STORE = ROOT/'dataset_store'
PAGE_OBJECTS = DATASET_STORE/'page_objects'
DATASET_VERSIONS = ROOT/'dataset_versions'
CURRENT_DATASET = DATASET_VERSIONS/'current_dataset.json'
PUBLISH_DATASET_VERSION = True

FORCE_REBUILD_ALL = False
REMOVE_OUTPUTS_FOR_DELETED_MASKS = True
MASK_NONZERO_THRESHOLD = 0
MIN_CLEAN_COMPONENT_AREA = 30
HITL_VALIDATION_RATE = 0.20

STANDARD_EXPANSION = {
    'dilation_size': int(CONFIG['training_mask']['dilation_size']),
    'dilation_iterations': int(CONFIG['training_mask']['dilation_iterations']),
    'closing_size': int(CONFIG['training_mask']['closing_size']),
}
HITL_EXPANSION = {
    'dilation_size': 3,
    'dilation_iterations': 1,
    'closing_size': 5,
}

TILE_SIZE = int(CONFIG['tile_size'])
TILE_STRIDE = int(CONFIG['tile_stride'])
LABEL = CONFIG['labels']
FIXED_VALIDATION = set(CONFIG.get('validation_pages', []))
SUPPORTED = {'.jpg','.jpeg','.png','.tif','.tiff'}

for folder in [ALIGNED_DIR,CORRECTED_MASK_DIR,TRAINING_MASK_DIR,TRAIN_IMAGES,VAL_IMAGES,TRAIN_LABELS,VAL_LABELS,TRAIN_MASK_TILES,VAL_MASK_TILES,CORRECTED_REVIEW,TRAINING_REVIEW,TILE_REVIEW,HITL_AUDIT_DIR,STATE_DIR,PAGE_OBJECTS,DATASET_VERSIONS]:
    folder.mkdir(parents=True, exist_ok=True)

print('Standard expansion:', STANDARD_EXPANSION)
print('HITL expansion:', HITL_EXPANSION)
print('HITL validation target:', HITL_VALIDATION_RATE)
print('Dataset version directory:', DATASET_VERSIONS)


## Identify HITL pages and create permanent split assignments

HITL pages are recognised from `human_in_the_middle/audit/Page###.json`. Assignments already stored are never changed. New or previously unassigned HITL pages are processed in page-number order, with every fifth HITL page assigned to validation.

This provides a stable 80/20 split. It does not solve source-set leakage by itself. If several pages are consecutive pages from the same score, later versions should add a `source_group` field to the audit and split by group.

In [ ]:
def image_files(folder):return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)
def find_image(folder,page):
 hits=[p for p in image_files(folder) if p.stem.lower()==page.lower()]
 return hits[0] if len(hits)==1 else None
def md5(path):
 h=hashlib.md5()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def object_hash(value):return hashlib.md5(json.dumps(value,sort_keys=True).encode()).hexdigest()
def read_json(path,default):
 try:return json.loads(Path(path).read_text()) if Path(path).exists() else default
 except Exception:return default
def write_json_atomic(path,value):
 temp=Path(str(path)+'.tmp');temp.write_text(json.dumps(value,indent=2));temp.replace(path)
def page_sort_key(page):
 m=re.search(r'(\d+)$',page);return int(m.group(1)) if m else 10**9

hitl_pages=sorted({p.stem for p in HITL_AUDIT_DIR.glob('Page*.json')},key=page_sort_key)
assignments=read_json(HITL_SPLIT_PATH,{'version':1,'assignments':{}})
split_map=assignments.get('assignments',{})
# Preserve existing assignments; allocate all new HITL pages in stable sequence.
assigned_hitl=sorted([p for p in hitl_pages if p in split_map],key=page_sort_key)
new_hitl=sorted([p for p in hitl_pages if p not in split_map],key=page_sort_key)
sequence_index=len(assigned_hitl)
for page in new_hitl:
 sequence_index+=1
 split_map[page]='val' if sequence_index%5==0 else 'train'
assignments={'version':2,'updated_utc':datetime.now(timezone.utc).isoformat(),'validation_rate':HITL_VALIDATION_RATE,'method':'persistent_every_fifth_hitl_page','assignments':split_map}
write_json_atomic(HITL_SPLIT_PATH,assignments)
print('HITL pages:',len(hitl_pages),'train:',sum(split_map.get(p)=='train' for p in hitl_pages),'val:',sum(split_map.get(p)=='val' for p in hitl_pages))
print('Split file:',HITL_SPLIT_PATH)

## Validate pages and calculate incremental fingerprints

In [ ]:
def normalise_corrected(mask):
 binary=np.where(mask>MASK_NONZERO_THRESHOLD,255,0).astype(np.uint8)
 n,labels,stats,_=cv2.connectedComponentsWithStats(binary,8);clean=np.zeros_like(binary)
 for i in range(1,n):
  if stats[i,cv2.CC_STAT_AREA]>=MIN_CLEAN_COMPONENT_AREA:clean[labels==i]=255
 return clean

def split_for(page):
 if page in split_map:return split_map[page]
 return 'val' if page in FIXED_VALIDATION else 'train'
def route_for(page):return 'hitl' if page in split_map else 'standard'
def expansion_for(page):return HITL_EXPANSION if route_for(page)=='hitl' else STANDARD_EXPANSION

def training_mask(mask,profile):
 result=mask.copy()
 if profile['dilation_size']>1 and profile['dilation_iterations']>0:
  kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(profile['dilation_size'],profile['dilation_size']))
  result=cv2.dilate(result,kernel,iterations=profile['dilation_iterations'])
 if profile['closing_size']>1:
  kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(profile['closing_size'],profile['closing_size']))
  result=cv2.morphologyEx(result,cv2.MORPH_CLOSE,kernel)
 return result

valid={};problems=[]
for mask_path in image_files(CORRECTED_MASK_DIR):
 page=mask_path.stem
 if not re.fullmatch(r'Page\d+',page,re.I):continue
 aligned_path=find_image(ALIGNED_DIR,page)
 if aligned_path is None:problems.append((page,'aligned image missing'));continue
 aligned=cv2.imread(str(aligned_path),0);raw=cv2.imread(str(mask_path),0)
 if aligned is None or raw is None or aligned.shape!=raw.shape:problems.append((page,'unreadable or shape mismatch'));continue
 mask=normalise_corrected(raw)  # Empty mask remains valid as a hard negative.
 split=split_for(page);route=route_for(page);profile=expansion_for(page)
 fingerprint={'aligned_md5':md5(aligned_path),'corrected_md5':md5(mask_path),'split':split,'route':route,'expansion':profile,'tile_size':TILE_SIZE,'tile_stride':TILE_STRIDE,'label_settings':LABEL,'minimum_clean_component_area':MIN_CLEAN_COMPONENT_AREA}
 valid[page]={'aligned_path':aligned_path,'corrected_path':mask_path,'mask':mask,'split':split,'route':route,'profile':profile,'fingerprint':fingerprint}
print('Valid pages:',len(valid),'Problems:',problems)
print('Train pages:',sum(v['split']=='train' for v in valid.values()),'Validation pages:',sum(v['split']=='val' for v in valid.values()))

## Rebuild changed pages only

In [ ]:
def positions(n):
 if n<=TILE_SIZE:return [0]
 p=list(range(0,n-TILE_SIZE+1,TILE_STRIDE))
 if p[-1]!=n-TILE_SIZE:p.append(n-TILE_SIZE)
 return p
def mask_to_lines(mask):
 contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE);lines=[];h,w=mask.shape
 for contour in contours:
  if cv2.contourArea(contour)<LABEL['minimum_contour_area']:continue
  poly=cv2.approxPolyDP(contour,max(1.0,LABEL['polygon_epsilon_fraction']*cv2.arcLength(contour,True)),True);values=[]
  for pt in poly:values.extend([pt[0][0]/w,pt[0][1]/h])
  if len(values)>=6:lines.append('0 '+' '.join(f'{v:.6f}' for v in values))
 return lines
def overlay(base,mask,alpha=.55):
 colour=cv2.cvtColor(base,cv2.COLOR_GRAY2BGR) if base.ndim==2 else base.copy();layer=colour.copy();layer[mask>0]=(0,0,255);return cv2.addWeighted(colour,1-alpha,layer,alpha,0)
def remove_page_outputs(page):
 for p in [TRAINING_MASK_DIR/f'{page}.png',CORRECTED_REVIEW/f'{page}.png',TRAINING_REVIEW/f'{page}.png']:
  if p.exists():p.unlink()
 for folder,pattern in [(TRAIN_IMAGES,f'{page}_y*_x*.jpg'),(VAL_IMAGES,f'{page}_y*_x*.jpg'),(TRAIN_MASK_TILES,f'{page}_y*_x*.png'),(VAL_MASK_TILES,f'{page}_y*_x*.png'),(TRAIN_LABELS,f'{page}_y*_x*.txt'),(VAL_LABELS,f'{page}_y*_x*.txt'),(TILE_REVIEW,f'*_{page}_y*_x*.jpg')]:
  for p in folder.glob(pattern):p.unlink()

state=read_json(STATE_PATH,{'pages':{}});old=state.get('pages',{});to_build=[];skipped=[]
def outputs_exist(page,split):
 idir=TRAIN_IMAGES if split=='train' else VAL_IMAGES;return (TRAINING_MASK_DIR/f'{page}.png').exists() and any(idir.glob(f'{page}_y*_x*.jpg'))
for page,info in valid.items():
 unchanged=(not FORCE_REBUILD_ALL and old.get(page,{}).get('fingerprint')==info['fingerprint'] and outputs_exist(page,info['split']))
 (skipped if unchanged else to_build).append(page)
print('Pages to rebuild:',to_build);print('Unchanged pages skipped:',len(skipped))
for page in to_build:remove_page_outputs(page)
if REMOVE_OUTPUTS_FOR_DELETED_MASKS:
 for page in sorted(set(old)-set(valid)):remove_page_outputs(page)

new_state={};records=[]
for page,info in valid.items():
 if page not in to_build:
  new_state[page]=old[page];continue
 mask=info['mask'];train_mask=training_mask(mask,info['profile']);cv2.imwrite(str(TRAINING_MASK_DIR/f'{page}.png'),train_mask)
 aligned_gray=cv2.imread(str(info['aligned_path']),0);aligned_colour=cv2.cvtColor(aligned_gray,cv2.COLOR_GRAY2BGR)
 cv2.imwrite(str(CORRECTED_REVIEW/f'{page}.png'),overlay(aligned_colour,mask,.65));cv2.imwrite(str(TRAINING_REVIEW/f'{page}.png'),overlay(aligned_colour,train_mask,.55))
 split=info['split'];idir=TRAIN_IMAGES if split=='train' else VAL_IMAGES;mdir=TRAIN_MASK_TILES if split=='train' else VAL_MASK_TILES;ldir=TRAIN_LABELS if split=='train' else VAL_LABELS;tile_count=positive=zero_polygon=0
 for y in positions(train_mask.shape[0]):
  for x in positions(train_mask.shape[1]):
   image_tile=aligned_colour[y:y+TILE_SIZE,x:x+TILE_SIZE];mask_tile=train_mask[y:y+TILE_SIZE,x:x+TILE_SIZE]
   if image_tile.shape[:2]!=(TILE_SIZE,TILE_SIZE):continue
   name=f'{page}_y{y:04d}_x{x:04d}';ip=idir/f'{name}.jpg';mp=mdir/f'{name}.png';lp=ldir/f'{name}.txt';cv2.imwrite(str(ip),image_tile,[cv2.IMWRITE_JPEG_QUALITY,95]);cv2.imwrite(str(mp),mask_tile);lines=mask_to_lines(mask_tile);lp.write_text('\n'.join(lines)+('\n' if lines else ''));tile_count+=1;is_positive=bool(np.any(mask_tile>0));positive+=int(is_positive);zero_polygon+=int(is_positive and not lines)
 new_state[page]={'fingerprint':info['fingerprint'],'processed_utc':datetime.now(timezone.utc).isoformat(),'split':split,'route':info['route'],'expansion':info['profile'],'tile_count':tile_count,'positive_tiles':positive,'positive_zero_polygon':zero_polygon}
 print('Rebuilt',page,split,info['route'],info['profile'],'tiles',tile_count)

## Publish immutable page objects and dataset releases

Each changed page becomes one uncompressed TAR object. Unchanged objects are reused.

In [ ]:
import tempfile

def page_object_identity(page, state_entry):
    return {
        'publication_schema': 2,
        'page': page,
        'fingerprint': state_entry['fingerprint'],
        'split': state_entry['split'],
        'route': state_entry['route'],
        'expansion': state_entry['expansion'],
        'tile_count': state_entry['tile_count'],
        'positive_tiles': state_entry['positive_tiles'],
        'positive_zero_polygon': state_entry['positive_zero_polygon'],
    }

def page_object_hash(page, state_entry):
    return object_hash(page_object_identity(page, state_entry))

def split_folders(split):
    if split == 'train':
        return {'images': TRAIN_IMAGES, 'labels': TRAIN_LABELS, 'mask_tiles': TRAIN_MASK_TILES}
    if split == 'val':
        return {'images': VAL_IMAGES, 'labels': VAL_LABELS, 'mask_tiles': VAL_MASK_TILES}
    raise ValueError(f'Unknown split: {split}')

def validate_page_files(page, split, expected):
    folders = split_folders(split)
    files = {kind: sorted(folder.glob(f'{page}_y*_x*.*')) for kind,folder in folders.items()}
    stems = {kind: {p.stem for p in paths} for kind,paths in files.items()}
    if stems['images'] != stems['labels'] or stems['images'] != stems['mask_tiles']:
        raise RuntimeError(f'{page}: image/label/mask mismatch before publication')
    if len(stems['images']) != expected:
        raise RuntimeError(f'{page}: expected {expected} tiles, found {len(stems["images"])}')
    return files

def publish_page_tar(page, state_entry):
    split = state_entry['split']
    object_hash_value = page_object_hash(page, state_entry)
    page_dir = PAGE_OBJECTS/page
    page_dir.mkdir(parents=True, exist_ok=True)
    object_path = page_dir/f'{object_hash_value}.tar'
    marker_path = page_dir/f'{object_hash_value}.json'
    if object_path.exists() and marker_path.exists():
        return object_hash_value, object_path, True

    files = validate_page_files(page, split, int(state_entry['tile_count']))
    temp_path = page_dir/f'.{object_hash_value}.tmp.tar'
    if temp_path.exists(): temp_path.unlink()
    with tarfile.open(temp_path, 'w') as archive:
        for kind, paths in files.items():
            for path in paths:
                archive.add(path, arcname=str(Path('dataset')/kind/split/path.name), recursive=False)
    with tarfile.open(temp_path, 'r') as archive:
        members = [m for m in archive.getmembers() if m.isfile()]
    expected_members = int(state_entry['tile_count']) * 3
    if len(members) != expected_members:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError(f'{page}: TAR contains {len(members)} files; expected {expected_members}')
    temp_path.replace(object_path)
    write_json_atomic(marker_path, {
        **page_object_identity(page, state_entry),
        'object_hash': object_hash_value,
        'relative_path': str(object_path.relative_to(ROOT)),
        'published_utc': datetime.now(timezone.utc).isoformat(),
    })
    return object_hash_value, object_path, False

def next_dataset_version():
    values=[]
    for path in DATASET_VERSIONS.glob('v*.json'):
        match=re.fullmatch(r'v(\d+)\.json', path.name)
        if match: values.append(int(match.group(1)))
    return max(values, default=0)+1


## Validate the accumulated dataset and refresh manifests

In [ ]:
train_pages = sorted(p for p,v in valid.items() if v['split']=='train')
val_pages = sorted(p for p,v in valid.items() if v['split']=='val')
all_records = []
for split,idir,mdir,ldir,pages in [('train',TRAIN_IMAGES,TRAIN_MASK_TILES,TRAIN_LABELS,train_pages),('val',VAL_IMAGES,VAL_MASK_TILES,VAL_LABELS,val_pages)]:
    images_map={p.stem:p for p in image_files(idir)}
    masks_map={p.stem:p for p in image_files(mdir)}
    labels_map={p.stem:p for p in ldir.glob('*.txt')}
    allowed=set(pages)
    for stem in sorted(set(images_map)|set(masks_map)|set(labels_map)):
        match=re.match(r'^(.*)_y\d+_x\d+$',stem)
        page=match.group(1) if match else None
        if page not in allowed:
            for collection in [images_map,masks_map,labels_map]:
                if stem in collection: collection[stem].unlink()
    images_map={p.stem:p for p in image_files(idir)}
    masks_map={p.stem:p for p in image_files(mdir)}
    labels_map={p.stem:p for p in ldir.glob('*.txt')}
    if set(images_map)!=set(masks_map) or set(images_map)!=set(labels_map):
        raise RuntimeError(f'{split} image/mask/label mismatch')
    for stem in sorted(images_map):
        mask=cv2.imread(str(masks_map[stem]),0)
        all_records.append({'tile_name':stem,'split':split,'positive':bool(np.any(mask>0)),'mask_pixels':int(np.sum(mask>0)),'polygon_count':sum(bool(line.strip()) for line in labels_map[stem].read_text().splitlines())})

YAML.write_text(f'path: {DATASET_DIR}\ntrain: images/train\nval: images/val\nnames:\n  0: annotation\n')
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'incremental':True,'training_pages':train_pages,'validation_pages':val_pages,'fixed_validation_pages':sorted(FIXED_VALIDATION),'hitl_split_file':str(HITL_SPLIT_PATH),'hitl_validation_rate':HITL_VALIDATION_RATE,'standard_expansion':STANDARD_EXPANSION,'hitl_expansion':HITL_EXPANSION,'pages':[{'page':p,'split':valid[p]['split'],'route':valid[p]['route'],'expansion':valid[p]['profile'],'aligned':str(valid[p]['aligned_path']),'corrected':str(valid[p]['corrected_path']),'training':str(TRAINING_MASK_DIR/f'{p}.png')} for p in train_pages+val_pages],'tiles':all_records}
MANIFEST.write_text(json.dumps(manifest,indent=2))
write_json_atomic(STATE_PATH,{'version':4,'updated_utc':datetime.now(timezone.utc).isoformat(),'pages':new_state})

if PUBLISH_DATASET_VERSION:
    page_objects=[];new_objects=0;reused_objects=0
    for page in sorted(new_state,key=page_sort_key):
        obj_hash,obj_path,reused=publish_page_tar(page,new_state[page])
        page_objects.append({'page':page,'split':new_state[page]['split'],'object_hash':obj_hash,'tile_count':int(new_state[page]['tile_count']),'relative_path':str(obj_path.relative_to(ROOT))})
        reused_objects += int(reused); new_objects += int(not reused)
    release_identity={'publication_schema':2,'pages':page_objects,'manifest_semantics':{'training_pages':train_pages,'validation_pages':val_pages,'fixed_validation_pages':sorted(FIXED_VALIDATION),'standard_expansion':STANDARD_EXPANSION,'hitl_expansion':HITL_EXPANSION},'config_snapshot':CONFIG}
    release_hash=object_hash(release_identity)
    current=read_json(CURRENT_DATASET,{})
    if current.get('release_hash')==release_hash:
        print('Dataset unchanged; retained release:',current.get('version_name'))
    else:
        number=next_dataset_version();version_name=f'v{number:06d}';release_path=DATASET_VERSIONS/f'{version_name}.json'
        train_tiles=sum(x['tile_count'] for x in page_objects if x['split']=='train')
        val_tiles=sum(x['tile_count'] for x in page_objects if x['split']=='val')
        release={'publication_schema':2,'version':number,'version_name':version_name,'release_hash':release_hash,'created_utc':datetime.now(timezone.utc).isoformat(),'manifest_snapshot':manifest,'manifest_md5':md5(MANIFEST),'config_snapshot':CONFIG,'config_md5':md5(CONFIG_PATH),'training_pages':train_pages,'validation_pages':val_pages,'training_tile_count':train_tiles,'validation_tile_count':val_tiles,'total_tile_count':train_tiles+val_tiles,'new_page_objects_in_this_run':new_objects,'reused_page_objects_in_this_run':reused_objects,'page_objects':page_objects}
        write_json_atomic(release_path,release)
        write_json_atomic(CURRENT_DATASET,{'publication_schema':2,'version':number,'version_name':version_name,'release_hash':release_hash,'release_file':str(release_path.relative_to(ROOT)),'updated_utc':datetime.now(timezone.utc).isoformat()})
        print('Published dataset:',version_name)
        print('New page objects:',new_objects,'Reused:',reused_objects)
        print('Training tiles:',train_tiles,'Validation tiles:',val_tiles)

print('Training pages:',len(train_pages));print('Validation pages:',len(val_pages));print('HITL train:',sum(valid[p]['route']=='hitl' for p in train_pages),'HITL val:',sum(valid[p]['route']=='hitl' for p in val_pages));print('Tiles:',len(all_records));print('DATASET READY FOR GPU TRAINING')
